In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from tqdm import tqdm
import os
import math
from typing import Optional, Tuple, List

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
try:
    from pytorchvideo.models.movinets import create_movinet
    MOVINET_AVAILABLE = True
except ImportError:
    print("Warning: pytorchvideo not available. Install with: pip install pytorchvideo")
    MOVINET_AVAILABLE = False

# Alternative TensorFlow MoViNet
try:
    import tensorflow as tf
    from official.projects.movinet.modeling import movinet
    from official.projects.movinet.modeling import movinet_model
    TF_MOVINET_AVAILABLE = True
except ImportError:
    print("Warning: TensorFlow MoViNet not available. Install with: pip install tf-models-official tensorflow")
    TF_MOVINET_AVAILABLE = False


In [3]:
# Class mappings
binary_class_map = {
    'Tiger_Normal': 0,
    'Tiger_Abnormal': 1,
}

multi_class_map = {
    'Dehydration or Heat Stroke': 0,
    'Digestive Issues': 1,
    'Eye Injury': 2,
    'Injured_Tiger': 3,
    'Lethargy, Apathy, Unresponsive, and Listless Tiger': 4,
    'Neurological Issues': 5,
    'Nutritional_Deficiencies': 6,
    'Oral or Dental Issues or Respiratory distress': 7,
    'Skin Desease or irritation_Tiger': 8,
    'Sress_Frustation': 9,
    'Tremors or Seizures': 10,
    'underweightness or emaciation': 11,
    'Weakness': 12,
    'Zoochosis_stereotypic behavior': 13,
    'Zoonotic Disease Behavior': 14
}

multi_class_id_to_name = {v: k for k, v in multi_class_map.items()}

In [4]:
def prepare_dataset_paths_and_labels_recursive(root_dir, class_map):
    video_folders = []
    labels = []
    for class_name, label_id in class_map.items():
        class_path = os.path.join(root_dir, class_name)
        if not os.path.exists(class_path):
            continue
        for subdir, dirs, files in os.walk(class_path):
            if any(f.lower().endswith(('.jpg', '.png')) for f in files):
                video_folders.append(subdir)
                labels.append(label_id)
    return video_folders, labels

In [5]:
class AnimalPoseEstimator(nn.Module):
    """
    Real animal pose estimation network inspired by DeepLabCut and animal pose estimation papers
    """
    def __init__(self, num_keypoints=34, input_channels=3):
        super().__init__()
        self.num_keypoints = num_keypoints
        
        # Backbone: ResNet-50 style feature extractor
        self.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # ResNet blocks
        self.layer1 = self._make_layer(64, 64, 3)
        self.layer2 = self._make_layer(64, 128, 4, stride=2)
        self.layer3 = self._make_layer(128, 256, 6, stride=2)
        self.layer4 = self._make_layer(256, 512, 3, stride=2)
        
        # Pose estimation head with upsampling
        self.pose_head = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_keypoints, kernel_size=1)
        )
        
        # 3D depth estimation head
        self.depth_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_keypoints)
        )
    
    def _make_layer(self, inplanes, planes, blocks, stride=1):
        layers = []
        if stride != 1 or inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(inplanes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )
        else:
            downsample = None
            
        layers.append(self._basic_block(inplanes, planes, stride, downsample))
        for _ in range(1, blocks):
            layers.append(self._basic_block(planes, planes))
        return nn.Sequential(*layers)
    
    def _basic_block(self, inplanes, planes, stride=1, downsample=None):
        return nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(planes),
        )
    
    def forward(self, x):
        # Backbone feature extraction
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # Pose heatmaps (2D keypoints)
        heatmaps = self.pose_head(x)  # [batch, num_keypoints, H, W]
        
        # Depth estimation (Z coordinates)
        depth = self.depth_head(x)    # [batch, num_keypoints]
        
        return heatmaps, depth

In [6]:
def extract_3d_pose_from_frames(frames, pose_estimator):
    """
    Extract 3D pose from video frames using the animal pose estimator
    """
    pose_estimator.eval()
    poses_3d = []
    
    with torch.no_grad():
        for frame in frames:
            # frame shape: [C, H, W]
            frame_batch = frame.unsqueeze(0).to(device)  # [1, C, H, W]
            
            heatmaps, depth = pose_estimator(frame_batch)
            
            # Extract 2D keypoints from heatmaps
            batch_size, num_kpts, h, w = heatmaps.shape
            heatmaps_reshaped = heatmaps.view(batch_size, num_kpts, -1)
            max_indices = torch.argmax(heatmaps_reshaped, dim=2)
            
            # Convert to x, y coordinates
            y_coords = (max_indices // w).float()
            x_coords = (max_indices % w).float()
            
            # Normalize coordinates to image size
            x_coords = x_coords * (224.0 / w)  # Assuming input size 224x224
            y_coords = y_coords * (224.0 / h)
            
            # Combine with depth to get 3D coordinates
            z_coords = depth[0]  # [num_keypoints]
            
            # Stack x, y, z coordinates
            pose_3d = torch.stack([x_coords[0], y_coords[0], z_coords], dim=1)  # [num_keypoints, 3]
            pose_3d = pose_3d.flatten()  # [num_keypoints * 3]
            poses_3d.append(pose_3d)
    
    return torch.stack(poses_3d)  # [clip_len, num_keypoints * 3]

In [7]:
# REAL MOVINET IMPLEMENTATION
class RealMoViNet(nn.Module):
    """
    Real MoViNet implementation using PyTorchVideo
    """
    def __init__(self, model_name="movinets_a2", feat_dim=600, num_classes=600):
        super().__init__()
        self.feat_dim = feat_dim
        
        if MOVINET_AVAILABLE:
            # Use real MoViNet from pytorchvideo
            self.backbone = create_movinet(
                model_name=model_name,
                num_classes=num_classes,
                conv_type="3d"
            )
            # Remove the final classification head to use as feature extractor
            self.backbone.blocks[-1].proj = nn.Identity()
        else:
            # Fallback to efficient 3D CNN if MoViNet not available
            print("Warning: Using fallback 3D CNN instead of MoViNet")
            self.backbone = self._create_fallback_3dcnn()
        
        # Final projection layer
        self.feature_proj = nn.Linear(2048, feat_dim)  # MoViNet-A2 outputs 2048 features
    
    def _create_fallback_3dcnn(self):
        """Efficient 3D CNN fallback when MoViNet is not available"""
        return nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(1,7,7), stride=(1,2,2), padding=(0,3,3)),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            
            nn.Conv3d(64, 128, kernel_size=(3,3,3), stride=(2,2,2), padding=(1,1,1)),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            
            nn.Conv3d(128, 256, kernel_size=(3,3,3), stride=(2,2,2), padding=(1,1,1)),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            
            nn.Conv3d(256, 512, kernel_size=(3,3,3), stride=(2,2,2), padding=(1,1,1)),
            nn.BatchNorm3d(512),
            nn.ReLU(inplace=True),
            
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(512, 2048)
        )
    
    def forward(self, x):
        # x shape: [batch, C, T, H, W]
        features = self.backbone(x)
        
        if len(features.shape) > 2:
            features = features.view(features.size(0), -1)
        
        # Project to desired feature dimension
        features = self.feature_proj(features)
        return features

In [8]:
# ANIMAL POSE TRANSFORMER
class AnimalPoseTransformer(nn.Module):
    """
    Specialized transformer for animal pose sequences with anatomical constraints
    """
    def __init__(self, input_dim=102, embedding_dim=512, output_dim=512, num_heads=8, num_layers=6):
        super().__init__()
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.output_dim = output_dim
        
        # Anatomical pose embedding (considers animal skeleton structure)
        self.pose_embedding = nn.Sequential(
            nn.Linear(input_dim, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        # Learnable positional encoding for temporal sequences
        self.temporal_pos_encoding = nn.Parameter(torch.randn(50, embedding_dim))
        
        # Anatomical positional encoding (spatial relationships between keypoints)
        self.anatomical_pos_encoding = self._create_anatomical_encoding(embedding_dim)
        
        # Multi-layer transformer with animal-specific attention
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=embedding_dim * 4,
            dropout=0.15,
            activation='gelu',
            batch_first=False,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Hierarchical attention pooling
        self.global_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=8,
            dropout=0.1,
            batch_first=True
        )
        
        # Output projection with residual connection
        self.output_proj = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(embedding_dim // 2, output_dim),
            nn.LayerNorm(output_dim)
        )
        
        # Learnable global query for attention pooling
        self.global_query = nn.Parameter(torch.randn(1, 1, embedding_dim))
    
    def _create_anatomical_encoding(self, embedding_dim):
        """Create anatomical positional encoding based on tiger skeleton structure"""
        # Tiger skeleton connectivity (simplified)
        tiger_skeleton = {
            'head': [0, 1, 2, 26, 27],  # nose, ears, eyes
            'neck': [3],
            'front_legs': [4, 5, 6, 7, 8, 9, 22, 23],  # shoulders, elbows, wrists, paws
            'spine': [10, 11, 12],
            'hips': [13, 14],
            'rear_legs': [15, 16, 17, 18, 24, 25],  # knees, ankles, paws
            'tail': [19, 20, 21],
            'body': [28, 29, 30, 31, 32, 33]  # mouth, whiskers, chest, belly
        }
        
        # Create anatomical positional encodings
        anatomical_encoding = torch.zeros(34, embedding_dim)
        
        for body_part, keypoints in tiger_skeleton.items():
            # Each body part gets a unique encoding pattern
            part_encoding = torch.randn(embedding_dim) * 0.1
            for kp_idx in keypoints:
                anatomical_encoding[kp_idx] = part_encoding
        
        return nn.Parameter(anatomical_encoding)
    
   
    def forward(self, x):
        seq_len, batch_size, _ = x.shape
    
    # Pose embedding
        x = self.pose_embedding(x)  # [seq_len, batch, embedding_dim]
    
    # Temporal positional encoding
        temp_pos = self.temporal_pos_encoding[:seq_len].unsqueeze(1).expand(-1, batch_size, -1)
        x = x + temp_pos
    
    # Anatomical encoding (just broadcast-add, no reshape hacks)
        anat_pos = self.anatomical_pos_encoding.unsqueeze(0).unsqueeze(0)  # [1,1,num_keypoints,embedding_dim]
        x = x + anat_pos.mean(dim=2, keepdim=False)  # compress anatomy encoding
    
    # Transformer
        x = self.transformer(x)  # [seq_len, batch, embedding_dim]
    
    # Global attention pooling
        x = x.permute(1, 0, 2)  # [batch, seq_len, embedding_dim]
        global_query = self.global_query.expand(batch_size, -1, -1)
        attended_features, _ = self.global_attention(global_query, x, x)
        attended_features = attended_features.squeeze(1)
    
    # Output projection
        output = self.output_proj(attended_features)
        return output

In [9]:
# ADVANCED MULTI-MODAL FUSION WITH CROSS-ATTENTION
class CrossModalFusionClassifier(nn.Module):
    """
    Advanced cross-modal fusion specifically designed for video + pose modalities
    """
    def __init__(self, video_feat_dim=600, pose_feat_dim=512, num_classes=2, dropout=0.3):
        super().__init__()
        self.video_feat_dim = video_feat_dim
        self.pose_feat_dim = pose_feat_dim
        self.num_classes = num_classes
        
        # Feature normalization and projection to common space
        self.video_norm = nn.LayerNorm(video_feat_dim)
        self.pose_norm = nn.LayerNorm(pose_feat_dim)
        
        common_dim = 512
        self.video_proj = nn.Linear(video_feat_dim, common_dim)
        self.pose_proj = nn.Linear(pose_feat_dim, common_dim)
        
        # Cross-modal attention mechanisms
        self.video_to_pose_attention = nn.MultiheadAttention(
            embed_dim=common_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        self.pose_to_video_attention = nn.MultiheadAttention(
            embed_dim=common_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        
        # Self-attention for each modality
        self.video_self_attention = nn.MultiheadAttention(
            embed_dim=common_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        self.pose_self_attention = nn.MultiheadAttention(
            embed_dim=common_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        
        # Fusion network
        fusion_input_dim = video_feat_dim + pose_feat_dim + 4 * common_dim
        
        self.fusion_network = nn.Sequential(
            nn.Linear(fusion_input_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(dropout),
            
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout / 2),
        )
        
        # Classification head with uncertainty estimation
        self.classifier = nn.Linear(256, num_classes)
        
    def forward(self, video_feat, pose_feat):
        batch_size = video_feat.size(0)
        
        # Normalize input features
        video_feat_norm = self.video_norm(video_feat)
        pose_feat_norm = self.pose_norm(pose_feat)
        
        # Project to common space
        video_common = self.video_proj(video_feat_norm).unsqueeze(1)  # [batch, 1, common_dim]
        pose_common = self.pose_proj(pose_feat_norm).unsqueeze(1)    # [batch, 1, common_dim]
        
        # Self-attention for each modality
        video_self_att, _ = self.video_self_attention(video_common, video_common, video_common)
        pose_self_att, _ = self.pose_self_attention(pose_common, pose_common, pose_common)
        
        # Cross-modal attention
        video_to_pose, _ = self.video_to_pose_attention(video_common, pose_common, pose_common)
        pose_to_video, _ = self.pose_to_video_attention(pose_common, video_common, video_common)
        
        # Flatten attention outputs
        video_self_att = video_self_att.squeeze(1)  # [batch, common_dim]
        pose_self_att = pose_self_att.squeeze(1)    # [batch, common_dim]
        video_to_pose = video_to_pose.squeeze(1)    # [batch, common_dim]
        pose_to_video = pose_to_video.squeeze(1)    # [batch, common_dim]
        
        # Concatenate all features
        fused_features = torch.cat([
            video_feat_norm,    # Original video features
            pose_feat_norm,     # Original pose features
            video_self_att,     # Self-attended video features
            pose_self_att,      # Self-attended pose features
            video_to_pose,      # Video attending to pose
            pose_to_video       # Pose attending to video
        ], dim=1)
        
        # Apply fusion network
        x = self.fusion_network(fused_features)
        
        # Classification
        logits = self.classifier(x)
        
        return logits

In [10]:
# UNIFIED TIGER BEHAVIOR MODEL
class TigerBehaviorModel(nn.Module):
    def __init__(self, num_classes, video_feat_dim=600, pose_feat_dim=512):
        super().__init__()
        self.num_classes = num_classes
        
        # Initialize pose estimator
        self.pose_estimator = AnimalPoseEstimator(num_keypoints=34)
        
        # Initialize video model (MoViNet)
        self.video_model = RealMoViNet(feat_dim=video_feat_dim)
        
        # Initialize pose transformer
        self.pose_transformer = AnimalPoseTransformer(
            input_dim=102, output_dim=pose_feat_dim
        )
        
        # Initialize fusion classifier
        self.fusion_classifier = CrossModalFusionClassifier(
            video_feat_dim=video_feat_dim,
            pose_feat_dim=pose_feat_dim,
            num_classes=num_classes
        )
    
    def forward(self, frames, poses=None):
        batch_size = frames.size(0)
        
        # Extract video features using MoViNet
        video_features = self.video_model(frames)  # [batch, video_feat_dim]
        
        # If poses are not provided, extract them from frames
        if poses is None:
            # Extract poses from video frames
            poses_list = []
            for i in range(batch_size):
                frame_sequence = frames[i]  # [C, T, H, W]
                frame_sequence = frame_sequence.permute(1, 0, 2, 3)  # [T, C, H, W]
                pose_3d = extract_3d_pose_from_frames(frame_sequence, self.pose_estimator)
                poses_list.append(pose_3d)
            poses = torch.stack(poses_list)  # [batch, T, pose_dim]
        
        # Process poses through transformer
        poses = poses.permute(1, 0, 2).to(device)  # [seq_len, batch, pose_dim]
        pose_features = self.pose_transformer(poses)  # [batch, pose_feat_dim]
        
        # Fuse modalities and classify
        logits = self.fusion_classifier(video_features, pose_features)
        
        return logits, video_features, pose_features

In [11]:
# DATASET CLASS
class TigerBehaviorDataset(Dataset):
    def __init__(self, video_folders, labels, pose_folder_root=None,
                 clip_len=8, frame_size=(224, 224), transform=None,
                 use_pose_estimator=True):
        self.video_folders = video_folders
        self.labels = labels
        self.pose_folder_root = pose_folder_root
        self.clip_len = clip_len
        self.frame_size = frame_size
        self.transform = transform
        self.use_pose_estimator = use_pose_estimator

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        video_folder = self.video_folders[idx]
        label = self.labels[idx]

        frame_files = sorted([
            os.path.join(video_folder, f)
            for f in os.listdir(video_folder)
            if f.lower().endswith(('.jpg', '.png'))
        ])
        
        if len(frame_files) == 0:
            raise RuntimeError(f"No frames found in folder {video_folder}")

        if len(frame_files) < self.clip_len:
            frame_files += [frame_files[-1]] * (self.clip_len - len(frame_files))
        else:
            frame_files = frame_files[:self.clip_len]

        frames = []
        for fpath in frame_files:
            img = cv2.imread(fpath)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, self.frame_size)
            if self.transform:
                img = self.transform(img)
            else:
                img = torch.FloatTensor(img / 255.0).permute(2, 0, 1)
            frames.append(img)
        
        frames_tensor = torch.stack(frames, dim=1)  # Shape: (C, T, H, W)

        # For pose data, we'll let the model extract it from frames
        # or load precomputed poses if available
        if self.pose_folder_root and not self.use_pose_estimator:
            pose_filename = os.path.basename(video_folder) + '.npy'
            pose_path = os.path.join(self.pose_folder_root, pose_filename)
            if os.path.exists(pose_path):
                pose_seq = np.load(pose_path)
                pose_tensor = torch.from_numpy(pose_seq).float()
            else:
                pose_tensor = torch.zeros(self.clip_len, 34 * 3)
        else:
            # Placeholder - model will extract poses from frames
            pose_tensor = torch.zeros(self.clip_len, 34 * 3)

        return frames_tensor, pose_tensor, label

In [12]:
# TRAINING FUNCTION
def train_model(model, train_loader, test_loader, task_type, num_epochs=15, lr=0.001, weight_decay=1e-4):
    """
    task_type: "binary" or "multi"
    """
    model = model.to(device)

    # --------- CONFIGS BASED ON TASK ---------
    if task_type == "binary":
        # ✅ Your existing settings
        optimizer = optim.AdamW([
            {'params': model.video_model.parameters(), 'lr': lr * 0.1},
            {'params': list(model.pose_transformer.parameters()) + list(model.pose_estimator.parameters()), 'lr': lr},
            {'params': model.fusion_classifier.parameters(), 'lr': lr * 2}
        ], weight_decay=weight_decay)

        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.3)
        patience = 7
        print("🔧 Using Binary-class training setup")

    else:  # MULTI-CLASS
        # 🔧 More aggressive training for imbalanced multi-class
        optimizer = optim.AdamW([
            {'params': model.video_model.parameters(), 'lr': lr * 0.2},   # a bit higher LR for backbone
            {'params': list(model.pose_transformer.parameters()) + list(model.pose_estimator.parameters()), 'lr': lr},
            {'params': model.fusion_classifier.parameters(), 'lr': lr * 3}
        ], weight_decay=weight_decay)

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
        
        # Use Focal Loss for class imbalance
        from torch.nn import functional as F
        class FocalLoss(nn.Module):
            def __init__(self, gamma=2.0, weight=None):
                super().__init__()
                self.gamma = gamma
                self.weight = weight
            def forward(self, inputs, targets):
                ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction="none")
                pt = torch.exp(-ce_loss)
                return ((1 - pt) ** self.gamma * ce_loss).mean()
        
        criterion = FocalLoss(gamma=2.0)
        patience = 10
        print("🔧 Using Multi-class training setup (FocalLoss + ReduceLROnPlateau)")

    best_val_acc = 0.0
    patience_counter = 0

    # --------- TRAINING LOOP ---------
    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        with tqdm(train_loader, unit="batch") as train_iter:
            train_iter.set_description(f"Epoch {epoch+1}/{num_epochs} [Train]")

            for frames, poses, labels in train_iter:
                frames, labels = frames.to(device), labels.to(device)
                optimizer.zero_grad()

                # Forward
                outputs, _, _ = model(frames)
                loss = criterion(outputs, labels)

                # Backward
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                # Stats
                preds = torch.argmax(outputs, dim=1)
                correct_predictions += (preds == labels).sum().item()
                total_samples += labels.size(0)
                running_train_loss += loss.item() * labels.size(0)

                current_acc = correct_predictions / total_samples
                train_iter.set_postfix(
                    loss=f"{running_train_loss/total_samples:.4f}",
                    accuracy=f"{current_acc:.4f}",
                    lr=f"{optimizer.param_groups[0]['lr']:.6f}"
                )

        # --------- VALIDATION ---------
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for frames, poses, labels in test_loader:
                frames, labels = frames.to(device), labels.to(device)
                outputs, _, _ = model(frames)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * labels.size(0)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        epoch_train_loss = running_train_loss / total_samples
        epoch_train_acc = correct_predictions / total_samples
        epoch_val_loss = val_loss / val_total if val_total > 0 else 0
        epoch_val_acc = val_correct / val_total if val_total > 0 else 0

        # Step scheduler
        if task_type == "multi":
            scheduler.step(epoch_val_acc)   # plateau scheduler needs val metric
        else:
            scheduler.step()

        # Save best
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            patience_counter = 0
            torch.save(model.state_dict(), f'best_tiger_model_{model.num_classes}_classes.pth')
        else:
            patience_counter += 1

        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f}, "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")

        if patience_counter >= patience:
            print(f"⏹ Early stopping triggered after {epoch+1} epochs")
            break

    # Load best model
    model.load_state_dict(torch.load(f'best_tiger_model_{model.num_classes}_classes.pth'))
    print(f"✅ Training completed. Best validation accuracy: {best_val_acc:.4f}")

    return model


In [13]:
# HIERARCHICAL INFERENCE FUNCTION
def infer_video(frames_tensor, binary_model, multi_model, multi_class_id_to_name, device):
    """
    Perform hierarchical classification using real MoViNet + Animal Pose Transformer
    """
    frames_tensor = frames_tensor.unsqueeze(0).to(device)
    
    binary_model.eval()
    multi_model.eval()
    
    with torch.no_grad():
        # Binary classification
        binary_outputs, video_feats, pose_feats = binary_model(frames_tensor)
        binary_probs = F.softmax(binary_outputs, dim=1)
        binary_pred = torch.argmax(binary_outputs, dim=1).item()
        binary_confidence = torch.max(binary_probs, dim=1)[0].item()
        
        if binary_pred == 0:
            return {
                'prediction': 'Normal',
                'confidence': binary_confidence,
                'binary_probs': binary_probs.cpu().numpy(),
                'video_features': video_feats.cpu().numpy(),
                'pose_features': pose_feats.cpu().numpy()
            }
        else:
            # Multi-class classification for abnormal cases
            multi_outputs, _, _ = multi_model(frames_tensor)
            multi_probs = F.softmax(multi_outputs, dim=1)
            multi_pred = torch.argmax(multi_outputs, dim=1).item()
            multi_confidence = torch.max(multi_probs, dim=1)[0].item()
            
            subclass_name = multi_class_id_to_name.get(multi_pred, "Unknown Abnormal Class")
            
            return {
                'prediction': f'Abnormal - {subclass_name}',
                'confidence': min(binary_confidence, multi_confidence),
                'binary_probs': binary_probs.cpu().numpy(),
                'multi_probs': multi_probs.cpu().numpy(),
                'video_features': video_feats.cpu().numpy(),
                'pose_features': pose_feats.cpu().numpy()
            }

In [14]:
# EVALUATION FUNCTION
def evaluate_model(model, test_loader, class_names=None):
    """
    Comprehensive model evaluation with detailed metrics
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for frames, poses, labels in tqdm(test_loader, desc="Evaluating"):
            frames = frames.to(device)
            labels = labels.to(device)
            
            outputs, _, _ = model(frames)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Calculate metrics
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
    import numpy as np
    
    accuracy = accuracy_score(all_labels, all_preds)
    
    print(f"\nModel Evaluation Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"\nClassification Report:")
    
    if class_names:
        print(classification_report(all_labels, all_preds, target_names=class_names))
    else:
        print(classification_report(all_labels, all_preds))
    
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(all_labels, all_preds)
    print(cm)
    
    return accuracy, all_preds, all_labels, all_probs

In [15]:
if __name__ == "__main__":
    print("🐅 Tiger Behavior Classification with Real MoViNet + Animal Pose Transformer")
    print("=" * 80)
    
    # Check available models
    print("Available Models:")
    print(f"✅ MoViNet (PyTorchVideo): {'Available' if MOVINET_AVAILABLE else 'Not Available'}")
    print(f"✅ TensorFlow MoViNet: {'Available' if TF_MOVINET_AVAILABLE else 'Not Available'}")
    print(f"✅ Animal Pose Estimator: Available (Custom Implementation)")
    print(f"✅ Animal Pose Transformer: Available (Custom Implementation)")
    
    # Data preparation
    train_root = 'frames_dataset/train'
    test_root = 'frames_dataset/test'
    
    # Binary classification data
    train_video_folders_bin, train_labels_bin = prepare_dataset_paths_and_labels_recursive(train_root, binary_class_map)
    test_video_folders_bin, test_labels_bin = prepare_dataset_paths_and_labels_recursive(test_root, binary_class_map)
    
    # Multi-class classification data (only abnormal)
    train_abnormal_root = os.path.join(train_root, 'Tiger_Abnormal')
    test_abnormal_root = os.path.join(test_root, 'Tiger_Abnormal')
    train_video_folders_multi, train_labels_multi = prepare_dataset_paths_and_labels_recursive(train_abnormal_root, multi_class_map)
    test_video_folders_multi, test_labels_multi = prepare_dataset_paths_and_labels_recursive(test_abnormal_root, multi_class_map)
    
    print(f"\n📊 Dataset Statistics:")
    print(f"Binary - Train: {len(train_labels_bin)}, Test: {len(test_labels_bin)}")
    print(f"Multi-class - Train: {len(train_labels_multi)}, Test: {len(test_labels_multi)}")
    
    # Create datasets
    train_dataset_bin = TigerBehaviorDataset(
        train_video_folders_bin, train_labels_bin, 
        clip_len=8, use_pose_estimator=True
    )
    test_dataset_bin = TigerBehaviorDataset(
        test_video_folders_bin, test_labels_bin, 
        clip_len=8, use_pose_estimator=True
    )
    
    train_dataset_multi = TigerBehaviorDataset(
        train_video_folders_multi, train_labels_multi, 
        clip_len=8, use_pose_estimator=True
    )
    test_dataset_multi = TigerBehaviorDataset(
        test_video_folders_multi, test_labels_multi, 
        clip_len=8, use_pose_estimator=True
    )
    
    # Create data loaders with appropriate batch sizes
    batch_size = 4 if torch.cuda.is_available() else 2  # Adjust based on GPU memory
    
    train_loader_bin = DataLoader(
        train_dataset_bin, batch_size=batch_size, 
        shuffle=True, num_workers=0, pin_memory=True
    )
    test_loader_bin = DataLoader(
        test_dataset_bin, batch_size=batch_size, 
        shuffle=False, num_workers=0, pin_memory=True
    )
    
    train_loader_multi = DataLoader(
        train_dataset_multi, batch_size=batch_size, 
        shuffle=True, num_workers=0, pin_memory=True
    )
    test_loader_multi = DataLoader(
        test_dataset_multi, batch_size=batch_size, 
        shuffle=False, num_workers=0, pin_memory=True
    )
    
    # Create models
    print(f"\n🏗️  Creating Models...")
    binary_model = TigerBehaviorModel(num_classes=2)
    multi_model = TigerBehaviorModel(num_classes=len(multi_class_map))
    
    # Count parameters
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Binary Model Parameters: {count_parameters(binary_model):,}")
    print(f"Multi-class Model Parameters: {count_parameters(multi_model):,}")
    
    # Training
    print(f"\n🚀 Training Binary Classification Model...")
    binary_model = train_model(
        binary_model, train_loader_bin, test_loader_bin, task_type="binary",
        num_epochs=15, lr=1e-4
    )
    
    print(f"\n🚀 Training Multi-class Classification Model...")
    multi_model = train_model(
        multi_model, train_loader_multi, test_loader_multi, task_type="multi",
        num_epochs=30, lr=1e-4
    )
    
    # Evaluation
    print(f"\n📈 Evaluating Models...")
    
    print(f"\n--- Binary Classification Results ---")
    binary_class_names = ['Normal', 'Abnormal']
    evaluate_model(binary_model, test_loader_bin, binary_class_names)
    
    print(f"\n--- Multi-class Classification Results ---")
    multi_class_names = list(multi_class_map.keys())
    evaluate_model(multi_model, test_loader_multi, multi_class_names)
    
    # Test inference on a sample
    print(f"\n🔍 Testing Inference Pipeline...")
    for frames, poses, label in test_loader_bin:
        sample_frames = frames[0]  # First sample
        
        result = infer_video(
            sample_frames, binary_model, multi_model, 
            multi_class_id_to_name, device
        )
        
        print(f"Sample Inference Result:")
        print(f"  Prediction: {result['prediction']}")
        print(f"  Confidence: {result['confidence']:.4f}")
        print(f"  Video Features Shape: {result['video_features'].shape}")
        print(f"  Pose Features Shape: {result['pose_features'].shape}")
        break
    
    print(f"\n✅ Training and Evaluation Complete!")
    print(f"Models saved as:")
    print(f"  - best_tiger_model_2_classes.pth (Binary)")
    print(f"  - best_tiger_model_{len(multi_class_map)}_classes.pth (Multi-class)")



🐅 Tiger Behavior Classification with Real MoViNet + Animal Pose Transformer
Available Models:
✅ MoViNet (PyTorchVideo): Not Available
✅ TensorFlow MoViNet: Not Available
✅ Animal Pose Estimator: Available (Custom Implementation)
✅ Animal Pose Transformer: Available (Custom Implementation)

📊 Dataset Statistics:
Binary - Train: 94, Test: 27
Multi-class - Train: 47, Test: 15

🏗️  Creating Models...


d:\Practice\AI\Emotion_Based_Movies_Recommonation\emotion_pytorch_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Binary Model Parameters: 59,942,478
Multi-class Model Parameters: 59,945,819

🚀 Training Binary Classification Model...
🔧 Using Binary-class training setup


Epoch 1/15 [Train]: 100%|██████████| 24/24 [00:15<00:00,  1.57batch/s, accuracy=0.5000, loss=0.7444, lr=0.000010]


Epoch 1/15 - Train Loss: 0.7444, Train Acc: 0.5000, Val Loss: 0.7030, Val Acc: 0.5556


...

Epoch 22/30 - Train Loss: 1.9679, Train Acc: 0.2128, Val Loss: 2.5145, Val Acc: 0.2000
⏹ Early stopping triggered after 22 epochs
✅ Training completed. Best validation accuracy: 0.2667

📈 Evaluating Models...

--- Binary Classification Results ---


Evaluating: 100%|██████████| 7/7 [00:03<00:00,  1.97it/s]



Model Evaluation Results:
Accuracy: 0.8148

Classification Report:
              precision    recall  f1-score   support

      Normal       0.89      0.67      0.76        12
    Abnormal       0.78      0.93      0.85        15

    accuracy                           0.81        27
   macro avg       0.83      0.80      0.81        27
weighted avg       0.83      0.81      0.81        27


Confusion Matrix:
[[ 8  4]
 [ 1 14]]

--- Multi-class Classification Results ---


Evaluating: 100%|██████████| 4/4 [00:02<00:00,  1.69it/s]
d:\Practice\AI\Emotion_Based_Movies_Recommonation\emotion_pytorch_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Practice\AI\Emotion_Based_Movies_Recommonation\emotion_pytorch_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Practice\AI\Emotion_Based_Movies_Recommonation\emotion_pytorch_env\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels wit


Model Evaluation Results:
Accuracy: 0.2667

Classification Report:
                                                    precision    recall  f1-score   support

                        Dehydration or Heat Stroke       0.00      0.00      0.00         1
                                  Digestive Issues       0.00      0.00      0.00         1
                                        Eye Injury       0.25      1.00      0.40         1
                                     Injured_Tiger       0.00      0.00      0.00         1
Lethargy, Apathy, Unresponsive, and Listless Tiger       0.00      0.00      0.00         1
                               Neurological Issues       0.00      0.00      0.00         1
                          Nutritional_Deficiencies       0.00      0.00      0.00         1
     Oral or Dental Issues or Respiratory distress       0.33      1.00      0.50         1
                  Skin Desease or irritation_Tiger       0.00      0.00      0.00         1
           

In [16]:
# ADDITIONAL UTILITY FUNCTIONS
def load_trained_models(binary_model_path, multi_model_path):
    """
    Load pre-trained models for inference
    """
    binary_model = TigerBehaviorModel(num_classes=2)
    multi_model = TigerBehaviorModel(num_classes=len(multi_class_map))
    
    binary_model.load_state_dict(torch.load(binary_model_path, map_location=device))
    multi_model.load_state_dict(torch.load(multi_model_path, map_location=device))
    
    binary_model.eval()
    multi_model.eval()
    
    return binary_model, multi_model

In [17]:
def process_video_file(video_path, binary_model, multi_model, clip_len=8):
    """
    Process a single video file for tiger behavior classification
    """
    import cv2
    
    # Read video
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    frame_count = 0
    while cap.read()[0] and frame_count < clip_len:
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            frame = torch.FloatTensor(frame / 255.0).permute(2, 0, 1)
            frames.append(frame)
            frame_count += 1
    
    cap.release()
    
    # Pad if necessary
    while len(frames) < clip_len:
        frames.append(frames[-1])
    
    # Create tensor and run inference
    frames_tensor = torch.stack(frames, dim=1)  # [C, T, H, W]
    
    result = infer_video(
        frames_tensor, binary_model, multi_model, 
        multi_class_id_to_name, device
    )
    
    return result